[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/20_clustering/04_dbscan.ipynb)

# Bài 04 — DBSCAN và HDBSCAN: phân cụm theo mật độ

**🎯 Mục tiêu**
- Ý tưởng mật độ: cụm = vùng **dày đặc** điểm, tách nhau bởi vùng thưa; điểm ở vùng thưa = **nhiễu**.
- Core point, border point, noise; tham số `eps`, `min_samples`; chọn `eps` bằng **k-distance plot**.
- DBSCAN from scratch (BFS mở rộng cụm) và scikit-learn.
- Failure: mật độ khác nhau → **HDBSCAN**; so sánh các thuật toán phân cụm.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/20_clustering

In [ ]:
from collections import deque

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN, HDBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import PALETTE, set_style

set_style()
rng = np.random.default_rng(0)


def color_of(labels):
    return ["lightgray" if l == -1 else PALETTE[l % len(PALETTE)] for l in labels]

## 1. Định nghĩa

Với bán kính $\varepsilon$ và ngưỡng `min_samples`:
- **Core point**: có ít nhất `min_samples` điểm (kể cả chính nó) trong bán kính $\varepsilon$.
- **Border point**: không phải core, nhưng nằm trong bán kính $\varepsilon$ của một core point.
- **Noise**: còn lại.

Cụm = tập core point **liên thông mật độ** (nối nhau qua chuỗi core point cách nhau ≤ ε) + border point của chúng. Không cần biết trước số cụm; cụm có **hình dạng tuỳ ý**.

## 2. From scratch

In [ ]:
def dbscan(X, eps, min_samples):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    neighbors = [np.flatnonzero(D[i] <= eps) for i in range(len(X))]
    core = np.array([len(nb) >= min_samples for nb in neighbors])
    labels = np.full(len(X), -1)
    cluster = 0
    for i in range(len(X)):
        if labels[i] != -1 or not core[i]:
            continue
        labels[i] = cluster                      # bắt đầu cụm mới từ một core point chưa gán
        queue = deque(neighbors[i])
        while queue:                             # BFS: mở rộng qua các core point
            j = queue.popleft()
            if labels[j] == -1:
                labels[j] = cluster
                if core[j]:
                    queue.extend(neighbors[j])
        cluster += 1
    return labels, core


Xm, ym = make_moons(400, noise=0.05, random_state=0)
Xm = np.vstack([Xm, rng.uniform([-1.5, -1], [2.5, 1.5], (30, 2))])           # + 30 điểm nhiễu
labels, core = dbscan(Xm, eps=0.15, min_samples=5)
sk = DBSCAN(eps=0.15, min_samples=5).fit_predict(Xm)
print(f"số cụm = {labels.max() + 1} | số điểm nhiễu = {(labels == -1).sum()} | khớp sklearn (ARI) = {adjusted_rand_score(sk, labels):.3f}")
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(*Xm[core].T, c=color_of(labels[core]), s=18, label="core")
ax.scatter(*Xm[~core].T, c=color_of(labels[~core]), s=18, marker="^", label="border / noise (xám)")
ax.legend()
ax.set_title("DBSCAN: hai mặt trăng + điểm nhiễu được gán −1")
plt.show()

## 3. Chọn eps: k-distance plot

Với `min_samples` = $m$, tính khoảng cách tới láng giềng thứ $m$ của mỗi điểm, sắp xếp tăng dần. Điểm trong cụm có khoảng cách nhỏ; điểm nhiễu có khoảng cách lớn → chọn $\varepsilon$ ở **"khuỷu"** nơi đường cong vọt lên.

In [ ]:
m = 5
dist, _ = NearestNeighbors(n_neighbors=m).fit(Xm).kneighbors(Xm)
kdist = np.sort(dist[:, -1])
plt.plot(kdist)
plt.axhline(0.15, color="r", ls="--", label="eps = 0.15")
plt.xlabel("điểm (sắp xếp)")
plt.ylabel(f"khoảng cách tới láng giềng thứ {m}")
plt.legend()
plt.title("k-distance plot")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.4))
for ax, eps in zip(axes, [0.05, 0.1, 0.15, 0.4]):
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xm)
    ax.scatter(*Xm.T, c=color_of(lab), s=8)
    ax.set_title(f"eps = {eps}: {lab.max() + 1} cụm, {(lab == -1).sum()} nhiễu")
plt.show()

- eps quá nhỏ: hầu hết là nhiễu, cụm vỡ vụn. eps quá lớn: mọi thứ gộp thành một cụm.
- `min_samples` lớn → yêu cầu mật độ cao hơn, nhiều nhiễu hơn; quy tắc kinh nghiệm: `min_samples` ≈ 2 × số chiều.
- Dùng khoảng cách → **scaling** cần thiết như KNN, K-Means.

## 4. Failure: mật độ khác nhau → HDBSCAN

Một eps duy nhất không thể phù hợp đồng thời với cụm dày và cụm thưa. **HDBSCAN** xây phân cấp theo mọi mức mật độ rồi chọn các cụm **bền vững** nhất — chỉ cần `min_cluster_size`.

In [ ]:
Xd = np.vstack([rng.normal([0, 0], 0.2, (300, 2)), rng.normal([3, 0], 0.2, (300, 2)), rng.normal([1.5, 3], 1.0, (300, 2))])
yd = np.repeat([0, 1, 2], 300)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, [("DBSCAN eps=0.2", DBSCAN(eps=0.2, min_samples=10)), ("DBSCAN eps=0.6", DBSCAN(eps=0.6, min_samples=10)),
                                    ("HDBSCAN min_cluster_size=30", HDBSCAN(min_cluster_size=30))]):
    lab = model.fit_predict(Xd)
    ax.scatter(*Xd.T, c=color_of(lab), s=6)
    ax.set_title(f"{name}: {lab.max() + 1} cụm, {(lab == -1).mean():.0%} nhiễu, ARI = {adjusted_rand_score(yd, lab):.2f}", fontsize=10)
plt.show()

eps nhỏ: cụm thưa bị coi là nhiễu; eps lớn: hai cụm dày (gần nhau) bị gộp. HDBSCAN xử lý được cả hai — thường là lựa chọn mặc định tốt hơn DBSCAN trong thực tế.

## 5. So sánh các thuật toán phân cụm

In [ ]:
data = {"blobs": make_blobs(500, centers=3, cluster_std=0.8, random_state=2),
        "mặt trăng": make_moons(500, noise=0.06, random_state=1),
        "vòng tròn": make_circles(500, factor=0.45, noise=0.05, random_state=1)}
algos = {"K-Means": lambda k: KMeans(k, n_init=10, random_state=0), "Ward": lambda k: AgglomerativeClustering(k),
         "Single linkage": lambda k: AgglomerativeClustering(k, linkage="single"), "DBSCAN": lambda k: DBSCAN(eps=0.2, min_samples=5),
         "HDBSCAN": lambda k: HDBSCAN(min_cluster_size=20)}
fig, axes = plt.subplots(3, 5, figsize=(24, 13))
for i, (dname, (Xk, yk)) in enumerate(data.items()):
    Xk = StandardScaler().fit_transform(Xk)
    k = len(np.unique(yk))
    for j, (aname, make) in enumerate(algos.items()):
        lab = make(k).fit_predict(Xk)
        axes[i, j].scatter(*Xk.T, c=color_of(lab), s=6)
        axes[i, j].set_title(f"{dname} — {aname}: ARI = {adjusted_rand_score(yk, lab):.2f}", fontsize=10)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
plt.tight_layout()
plt.show()

| Thuật toán | Cần K? | Hình dạng cụm | Nhiễu | Quy mô | Ghi chú |
|---|---|---|---|---|---|
| K-Means | có | lồi, đẳng hướng | không | rất lớn | nhanh, đơn giản |
| GMM (chương 23) | có | ellipse | không (xác suất) | lớn | soft clustering |
| Hierarchical (Ward) | không (cắt cây) | gọn | không | nhỏ–vừa | dendrogram |
| Single linkage | không | tuỳ ý | nhạy (chaining) | nhỏ–vừa | |
| DBSCAN | không | tuỳ ý | **có** | vừa–lớn | một mức mật độ |
| **HDBSCAN** | không | tuỳ ý | **có** | vừa–lớn | nhiều mức mật độ, ít tham số |

## 🧠 Kiểm tra nhanh

1. Phân biệt core point, border point, noise.
2. Chọn eps bằng k-distance plot thế nào?
3. Vì sao DBSCAN thất bại khi các cụm có mật độ rất khác nhau? HDBSCAN khắc phục thế nào?
4. Dữ liệu 500 000 giao dịch, muốn tìm nhóm hành vi và giao dịch bất thường. Thuật toán nào?

## 📌 Tóm tắt

- DBSCAN: cụm = vùng dày đặc liên thông; tự phát hiện nhiễu; không cần K; tham số eps, min_samples (k-distance plot).
- Thất bại khi mật độ khác nhau → HDBSCAN.
- Chọn thuật toán theo hình dạng cụm mong đợi, nhiễu, quy mô, nhu cầu diễn giải.

## 📚 Tài liệu tham khảo

- Ester et al. (1996) — *A Density-Based Algorithm for Discovering Clusters in Large Spatial Databases with Noise*
- Campello, Moulavi & Sander (2013) — *Density-Based Clustering Based on Hierarchical Density Estimates* (HDBSCAN)
- scikit-learn User Guide — Clustering (bảng so sánh thuật toán)